# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/darksider747/flyrank-1st/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes
Signal 1: CTR vs position — CONFIRMED
I checked whether CTR varies meaningfully by position tier, using the
starter dataset. Mean CTR drops sharply from page_1 (0.652) to deep
positions (0.150), based on [n from page_1] and [n from deep] pages
respectively. This confirms the assumption behind FlyRank's real
needs_ctr_fix logic: pages ranked worse genuinely do get fewer clicks,
so a low CTR should be judged relative to a page's position, not in
isolation.
 Signal 2: staleness (days_since_last_update / freshness_tier) — MIXED

I checked whether staler pages decline more often, using freshness_tier
buckets. Decline rate rises from 51.1% (0-30 days, n=20,480) to 58.9%
(31-90 days, n=175) to 61.1% (91-180 days, n=9,171) - supporting the
staleness assumption, and these numbers are backed by large, trustworthy
sample sizes for the 0-30 and 91-180 buckets.

However, the 181+ bucket breaks the pattern, showing only 47.1% decline
rate - but this bucket has only 174 pages, a small enough sample that
this result could easily be due to chance rather than a real reversal
of the staleness trend.

Verdict: MIXED. Staleness shows a real, meaningful relationship with
decline for well-populated buckets, but the pattern isn't perfectly
clean at the extreme end where sample sizes are too small to trust.
I will still use staleness in my rule, but I will treat the "very stale"
threshold cautiously given the thin data there.

My rule: flag a page as needing review if it has enough traffic to matter,
AND either its CTR is meaningfully below average for its position tier,
OR it hasn't been updated in 91-180+ days (the range where staleness
showed a real, trustworthy relationship with decline).

Reason codes:
- "ctr_below_tier_average": CTR is notably lower than other pages at the
  same position
- "stale_content": hasn't been updated in 91+ days
- "ctr_and_stale": both signals present (highest priority - gets the
  highest score)

Action label:
- "review_title_meta" - if flagged mainly for low CTR
- "review_content_refresh" - if flagged mainly for staleness  
- "high_priority_review" - if flagged for both

In [11]:
import os, subprocess

REPO_URL = "https://github.com/darksider747/flyrank-1st"
REPO_DIR = "flyrank-1st"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Now in:", os.getcwd())

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df)} rows")

# Signal 1: CTR vs position
ctr_by_tier = df.groupby("position_tier")["ctr"].agg(["mean", "count"])
print(ctr_by_tier)

# Signal 2: staleness
staleness_check = df.groupby("freshness_tier")["trend_direction"].apply(lambda x: (x == "down").mean())
staleness_count = df.groupby("freshness_tier")["trend_direction"].count()
staleness_table = pd.DataFrame({"decline_rate": staleness_check, "n": staleness_count})
print(staleness_table)

Now in: /content/flyrank-1st/flyrank-1st/flyrank-1st/flyrank-1st
Loaded 30000 rows
                   mean  count
position_tier                 
deep           0.150212   1319
page_1         0.652467  11814
page_3_5       0.222484   7242
striking       0.323239   7304
top_3          1.483611   2321
                decline_rate      n
freshness_tier                     
0-30                0.511377  20480
181+                0.471264    174
31-90               0.588571    175
91-180              0.611057   9171


I encoded my rule as a simple additive score:
- ctr_below_tier (1 point if CTR is meaningfully below the page's own
  position tier average, using a 30% threshold)
- is_stale (1 point if not updated in 91+ days, based on my staleness
  signal check)

Score = sum of these two flags (0, 1, or 2).

Reason code and action breakdown across all 30,000 pages:
- ctr_and_stale (score=2): 7,016 pages -> high_priority_review
- ctr_below_tier_average (score=1): 15,712 pages -> review_title_meta
- stale_content (score=1): 2,329 pages -> review_content_refresh
- no_flag (score=0): 4,943 pages -> monitor

The ranked queue was written to work/outputs/baseline_action_score.csv,
sorted by score descending.

Limitation: because the score only has 3 possible values (0, 1, 2),
thousands of pages tie at the same score - meaning the exact order
among tied pages is somewhat arbitrary, not a fine-grained ranking.
A more advanced model (coming in later weeks) would produce a smoother,
more precisely ordered score.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

# Get each page's tier-average CTR, so we can compare it fairly to peers
tier_avg_ctr = df.groupby("position_tier")["ctr"].transform("mean")
df["ctr_below_tier"] = df["ctr"] < (0.7 * tier_avg_ctr)  # meaningfully below tier average

df["is_stale"] = df["days_since_last_update"] >= 91

# Reason code logic
def get_reason_code(row):
    if row["ctr_below_tier"] and row["is_stale"]:
        return "ctr_and_stale"
    elif row["ctr_below_tier"]:
        return "ctr_below_tier_average"
    elif row["is_stale"]:
        return "stale_content"
    else:
        return "no_flag"

df["reason_code"] = df.apply(get_reason_code, axis=1)

# Action label logic
def get_action(reason):
    if reason == "ctr_and_stale":
        return "high_priority_review"
    elif reason == "ctr_below_tier_average":
        return "review_title_meta"
    elif reason == "stale_content":
        return "review_content_refresh"
    else:
        return "monitor"

df["action"] = df["reason_code"].apply(get_action)

# Score: combine both signals into one number (0 to 2)
df["score"] = df["ctr_below_tier"].astype(int) + df["is_stale"].astype(int)

print(df["reason_code"].value_counts())
print(df["action"].value_counts())

os.makedirs("work/outputs", exist_ok=True)

ranked = df.sort_values("score", ascending=False)
ranked[["content_id", "score", "reason_code", "action", "ctr", "position_tier", "days_since_last_update"]].to_csv(
    "work/outputs/baseline_action_score.csv", index=False
)

print("Saved ranked queue with", len(ranked), "rows")
ranked.head(10)

reason_code
ctr_below_tier_average    15712
ctr_and_stale              7016
no_flag                    4943
stale_content              2329
Name: count, dtype: int64
action
review_title_meta         15712
high_priority_review       7016
monitor                    4943
review_content_refresh     2329
Name: count, dtype: int64
Saved ranked queue with 30000 rows


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,ctr_below_tier,is_stale,reason_code,action,score
25030,content_47757540a16b,client_8527a891e2,0.0,0.00,LOW,0.00,keyword article,informational,4022.0,23297.0,...,0.0,low,page_1,flat,NaN,True,True,ctr_and_stale,high_priority_review,2
25029,content_351051794a7b,client_3fdba35f04,10.0,0.43,MEDIUM,0.00,keyword article,transactional,1378.0,8688.0,...,0.0,low,striking,down,-78.5,True,True,ctr_and_stale,high_priority_review,2
29982,content_fe5d259e6bc5,client_19581e27de,20.0,0.10,LOW,0.12,keyword article,transactional,NaN,NaN,...,0.0,moderate,page_3_5,up,74.0,True,True,ctr_and_stale,high_priority_review,2
24,content_0e23e310d404,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.0,good,page_1,down,-20.7,True,True,ctr_and_stale,high_priority_review,2
16,content_78bd1d4a1d4d,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,8200.0,52393.0,...,0.0,good,page_1,down,-39.1,True,True,ctr_and_stale,high_priority_review,2
46,content_c59d46264834,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.0,moderate,page_1,down,-47.5,True,True,ctr_and_stale,high_priority_review,2
39,content_4595e8704e07,client_8527a891e2,90.0,0.06,LOW,0.03,keyword article,informational,3666.0,21824.0,...,0.0,low,page_3_5,down,-100.0,True,True,ctr_and_stale,high_priority_review,2
25011,content_cc93422aa553,client_3fdba35f04,10.0,0.14,LOW,0.03,keyword article,informational,1810.0,11060.0,...,0.0,moderate,page_3_5,down,-45.8,True,True,ctr_and_stale,high_priority_review,2
25009,content_8727cbf06ba9,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,transactional,NaN,NaN,...,0.0,good,page_1,down,-33.4,True,True,ctr_and_stale,high_priority_review,2
25008,content_e821bd4f8ee7,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.0,low,striking,up,27.5,True,True,ctr_and_stale,high_priority_review,2


## 3. Top-20 review

Top-20 Review

1. content_47757540a16b — flagged: high priority (bad CTR + old)
   Why flagged: gets 0 clicks even though it's ranked #1 (page_1), and
   hasn't been touched in 104 days.
   Why it might be a mistake: it only got seen 5 times total. With that
   few views, getting 0 clicks could just be bad luck, not a real problem.

2. content_351051794a7b — flagged: high priority
   Why flagged: 0 clicks, old page.
   Why it might be a mistake: only 225 views — still low enough that
   0 clicks might just be chance.

3. content_fe5d259e6bc5 — flagged: high priority
   Why flagged: 0 clicks, old page, but this one had 1,573 views — a
   decent amount of traffic, so the 0 clicks means more this time.
   Why it might be a mistake: pages at this ranking spot (page_3_5)
   naturally get fewer clicks anyway, so 0 might not be as unusual here
   as it would be for a #1-ranked page.

4. content_0e23e310d404 — flagged: high priority
   Why flagged: only 27% of viewers click, but this page got seen
   29,541 times — that's a LOT of people not clicking.
   Why it might be a mistake: maybe 27% is actually normal for this
   type of page/topic, and I'm unfairly comparing it to a general average.

5. content_78bd1d4a1d4d — flagged: high priority
   Why flagged: only 15% click rate at the #1 spot, with strong traffic
   (13,848 views) — this is a solid, trustworthy problem.
   Why it might be a mistake: honestly, hard to find a flaw here — good
   traffic, clear gap. This looks like a genuinely correct pick.

6. content_c59d46264834 — flagged: high priority
   Why flagged: 8% click rate at #1 spot, decent traffic (1,193 views).
   Why it might be a mistake: traffic is okay but not huge — still
   probably a real problem, just lower priority than bigger pages.

7. content_4595e8704e07 — flagged: high priority
   Why flagged: 0 clicks, old page.
   Why it might be a mistake: only got seen 4 times — basically no data
   at all, so this flag doesn't mean much.

8. content_cc93422aa553 — flagged: high priority
   Why flagged: 11% click rate, 1,904 views, old page.
   Why it might be a mistake: this ranking spot naturally gets fewer
   clicks anyway, so 11% might be close to normal here.

9. content_8727cbf06ba9 — flagged: high priority
   Why flagged: 26% click rate at #1 spot with 5,341 views — meaningful
   traffic and a real gap from what's normal.
   Why it might be a mistake: 26% isn't terrible — a person checking
   this page might find it's actually fine, just a bit below average.

10. content_e821bd4f8ee7 — flagged: high priority
    Why flagged: 0 clicks, old page.
    Why it might be a mistake: only 267 views — still fairly low, so
    0 clicks could be chance.

11. content_d1dc90c8a4f9 — flagged: high priority
    Why flagged: 0 clicks at #1 spot, old page.
    Why it might be a mistake: only 60 views total — way too little
    data to trust "0 clicks" as meaningful.

12. content_c3111dcf8bec — flagged: high priority
    Why flagged: 7% click rate at #1 spot, 6,088 views — solid traffic,
    clear real problem.
    Why it might be a mistake: hard to find a flaw — good sample size,
    clear gap, high-value page.

13. content_09ed1d6745c5 — flagged: high priority
    Why flagged: 0 clicks at #1 spot, old page.
    Why it might be a mistake: only 8 views — almost no data, probably
    not a real signal.

14. content_52b2d29eb909 — flagged: high priority
    Why flagged: 38% click rate at #1 spot, 6,518 views.
    Why it might be a mistake: 38% is only a little below normal, not
    dramatically bad — maybe shouldn't be ranked as "high priority"
    alongside pages with 0% click rate.

15. content_3bcf2b902283 — flagged: high priority
    Why flagged: 34% click rate, 4,667 views.
    Why it might be a mistake: similar to #14 — the gap here is small,
    not severe, so treating it the same as a 0%-click page may be unfair.

16. content_532855f29974 — flagged: high priority
    Why flagged: 0 clicks, old page.
    Why it might be a mistake: only 87 views — too little data to trust.

17. content_47b970493481 — flagged: high priority
    Why flagged: 0 clicks, old page.
    Why it might be a mistake: only 153 views — still fairly low.

18. content_bea03a08f12c — flagged: high priority
    Why flagged: 8% click rate at #1 spot, 1,213 views.
    Why it might be a mistake: traffic is okay but not huge — real
    problem, just less urgent than the bigger pages.

19. content_f30d30832c2d — flagged: high priority
    Why flagged: 16% click rate at #1 spot, 644 views.
    Why it might be a mistake: lower traffic — the problem is real,
    but affects fewer total people than the bigger pages on this list.

20. content_5a6a88d83124 — flagged: high priority
    Why flagged: 0 clicks, but this time with 2,387 views — a decent
    amount of traffic, so this 0% is more trustworthy than the other
    "0 click" rows above.
    Why it might be a mistake: hard to find a flaw here — the sample
    size makes this a solid pick.

One big thing I noticed: every single one of these 20 pages shows the
exact same "104 days since last update." That's suspicious — 20
different pages all being updated on the exact same day seems unlikely
to happen by pure chance. This is probably some kind of data quirk
(like a placeholder value used when the real update date is missing),
not a real pattern. Worth mentioning as a limitation, not treating it
as solid proof of staleness.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = ranked.head(20)[["content_id", "score", "reason_code", "action", "ctr", "position_tier", "days_since_last_update", "impressions_90d"]]
print(top20.to_string())

                 content_id  score    reason_code                action   ctr position_tier  days_since_last_update  impressions_90d
25030  content_47757540a16b      2  ctr_and_stale  high_priority_review  0.00        page_1                     104                5
25029  content_351051794a7b      2  ctr_and_stale  high_priority_review  0.00      striking                     104              225
29982  content_fe5d259e6bc5      2  ctr_and_stale  high_priority_review  0.00      page_3_5                     104             1573
24     content_0e23e310d404      2  ctr_and_stale  high_priority_review  0.27        page_1                     104            29541
16     content_78bd1d4a1d4d      2  ctr_and_stale  high_priority_review  0.15        page_1                     104            13848
46     content_c59d46264834      2  ctr_and_stale  high_priority_review  0.08        page_1                     104             1193
39     content_4595e8704e07      2  ctr_and_stale  high_priority_revi

Weak picks:
Looking across my top 20, the weakest picks are the ones with very low
impressions_90d - under ~270 views (content_47757540a16b,
content_351051794a7b, content_4595e8704e07, content_e821bd4f8ee7,
content_d1dc90c8a4f9, content_09ed1d6745c5, content_532855f29974,
content_47b970493481). These show "0.00 CTR," but with so few views,
that number could easily be chance rather than a real problem. My
simple score treats these the same as high-traffic pages with the same
score - a real weakness, since it ignores sample size and traffic volume.

Leakage check:
My rule only uses ctr, position_tier, and days_since_last_update - all
signals I would genuinely know today, at decision time. I did not use
any future-window data, and I did not use any of FlyRank's actual
product decision flags (health_score, priority_score, etc.) - these
aren't even present in my dataset. This matters because this baseline
becomes the bar my Week 5 model needs to honestly beat.

Data-quality note (not leakage, but worth flagging): every top-20 row
shows days_since_last_update = 104 identically - likely a data quirk,
not a real coincidence. I'm treating staleness cautiously because of this.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Weak picks check - confirm which top-20 rows have low impressions
weak_threshold = 300
weak_picks = top20[top20["impressions_90d"] < weak_threshold]
print(f"Weak picks (impressions_90d < {weak_threshold}): {len(weak_picks)} of top 20")
print(weak_picks[["content_id", "impressions_90d", "ctr"]])

# Leakage check - confirm the exact columns used to build the score
features_used = ["ctr", "position_tier", "days_since_last_update"]
print("\nFeatures used in rule:", features_used)

unsafe_columns = [c for c in df.columns if "health_score" in c.lower() or "priority_score" in c.lower() or "action_type" in c.lower()]
print("Any unsafe product-flag columns present in dataset:", unsafe_columns if unsafe_columns else "None found")

Weak picks (impressions_90d < 300): 8 of top 20
                 content_id  impressions_90d  ctr
25030  content_47757540a16b                5  0.0
25029  content_351051794a7b              225  0.0
39     content_4595e8704e07                4  0.0
25008  content_e821bd4f8ee7              267  0.0
25037  content_d1dc90c8a4f9               60  0.0
25050  content_09ed1d6745c5                8  0.0
24976  content_532855f29974               87  0.0
25005  content_47b970493481              153  0.0

Features used in rule: ['ctr', 'position_tier', 'days_since_last_update']
Any unsafe product-flag columns present in dataset: None found


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.